In [1]:
"""
Script Name: load_capacidad_energetica.py
Authores: Walter Arredondo, Carlos Daza, Andrés Coral
Fecha: 2026-10-09
Fase ETL: Carga
Descripción:
Carga y unifica los 2 archivos procedentes de tranform
Entradas: - transform_energia_sostenible_WB_SE4ALL.csv y
transform_HUB_energia.csv
Salidas: - Archivo CSV Unificado: data/processed/energia.csv
- Archivo de la base de datos: data/processed/energia.db
Notas: - Ejecutar este script una sola vez
"""

'\nScript Name: load_capacidad_energetica.py\nAuthores: Walter Arredondo, Carlos Daza, Andrés Coral\nFecha: 2026-10-09\nFase ETL: Carga\nDescripción:\nCarga y unifica los 2 archivos procedentes de tranform\nEntradas: - transform_energia_sostenible_WB_SE4ALL.csv y\ntransform_HUB_energia.csv\nSalidas: - Archivo CSV Unificado: data/processed/energia.csv\n- Archivo de la base de datos: data/processed/energia.db\nNotas: - Ejecutar este script una sola vez\n'

In [2]:
import pandas as pd

In [3]:
from google.colab import drive

# Montar Google Drive
drive.mount('/content/drive')

Mounted at /content/drive


Se carga el Dataset  transform_energia_sostenible_WB_SE4ALL.csv

In [4]:
ruta = "/content/drive/My Drive/Colab Notebooks/Master/1.ETL/entregable3/data/transformed/transform_energia_sostenible_WB_SE4ALL.csv"
df1 = pd.read_csv(ruta, sep = ';')

Se carga el Dataset  transform_HUB_energia.csv

In [5]:
ruta = "/content/drive/My Drive/Colab Notebooks/Master/1.ETL/entregable3/data/transformed/transform_HUB_energia.csv"
df2 = pd.read_csv(ruta, sep = ';')

# Merge

In [6]:
# Eliminar posibles espacios en blanco en los nombres de las columnas
df1.columns = df1.columns.str.strip()
df2.columns = df2.columns.str.strip()

In [7]:
#Unir ambos dataframes por País y Año
df_final = pd.merge(df1, df2, on=['pais', 'anio'], how='left')

# Salida del Merge
Se guardarán los datos del Merge en un csv

In [8]:
ruta = "/content/drive/My Drive/Colab Notebooks/Master/1.ETL/entregable3/data/processed/"

# index = False; evita guardar el índice de pandas como una columna adicional
df_final.to_csv(ruta + "energia.csv", sep = ";", index = False)
print(df_final.shape)

(440, 48)


In [9]:
import re
import sqlite3
import unicodedata
from pathlib import Path

In [10]:
ruta = "/content/drive/My Drive/Colab Notebooks/Master/1.ETL/entregable3/data/processed/"
CSV = ruta + "energia.csv"
DB = ruta +  "energia.db"

In [11]:
# Cargar el csv unificado en un dataframe
df = pd.read_csv(CSV, sep=";", encoding="utf-8")

## 2. Crear el esquema

Se procede a crear el esquema de la base de datos con sus respectivas tablas para tener persistencia del dataset maestro energia.csv. Para guardar todos los datos se crean 8 tablas que son:



*   **dim_pais:** almacena el catálogo de países incluidos en la base de datos. Su función principal es identificar de manera única cada país y servir como referencia para relacionar la información energética registrada en las demás tablas.
*   **hechos_capacidad_renovable:** registra la capacidad instalada de energía renovable por habitante para cada país y año. Permite analizar la disponibilidad de capacidad renovable per cápita y su distribución según las diferentes fuentes de energía.
*   **hechos_capacidad:** contiene información sobre la capacidad energética instalada de cada país por año, desglosada según distintas tecnologías y fuentes de generación. A diferencia de la tabla anterior, sus variables no están expresadas por habitante, sino que representan la capacidad de cada categoría energética.
*   **hechos_consumo_renovable:** almacena los datos del consumo de energía renovable por país y año, clasificados según el sector o uso energético. Contempla el consumo de renovables para la generación de electricidad, la producción de calor y el transporte.
*   **hechos_consumo:** registra el consumo final de energía de cada país por año, desglosado por sectores económicos y sociales. Incluye tanto actividades productivas como usos residenciales y de transporte, además del consumo final total.
*   **hechos_generacion:** almacena los datos de generación de energía eléctrica por país y año, clasificados según la fuente de producción. Permite distinguir la contribución de las tecnologías renovables, no renovables y nuclear al suministro energético.
*   **hechos_financieros:** registra los flujos financieros destinados a las energías renovables en cada país y año. La información se distribuye por tecnología, lo que permite estudiar la financiación recibida por distintas fuentes renovables y su evolución temporal.
*   **hechos_otros:** reúne indicadores complementarios sobre el acceso a los servicios energéticos, la participación de las energías renovables y la eficiencia energética de cada país por año. Su finalidad es ofrecer una perspectiva más amplia de la sostenibilidad y el desarrollo energético.

La clave primaria de dim_pais es id_pais que es un campo autoincremental para identificar los paises con mayor facilidad. El resto de las tablas tienen una clave primaria compuesta por id_pais y anio para tener registros anuales de cada país.

In [12]:
con = sqlite3.connect(DB)

con.executescript(
    """
    PRAGMA foreign_keys = ON;

    DROP TABLE IF EXISTS hechos_capacidad_renovable;
    DROP TABLE IF EXISTS hechos_capacidad;
    DROP TABLE IF EXISTS hechos_consumo_renovable;
    DROP TABLE IF EXISTS hechos_consumo;
    DROP TABLE IF EXISTS hechos_generacion;
    DROP TABLE IF EXISTS hechos_financieros;
    DROP TABLE IF EXISTS hechos_otros;
    DROP TABLE IF EXISTS dim_pais;

    CREATE TABLE dim_pais (
        id_pais INTEGER PRIMARY KEY AUTOINCREMENT,
        pais TEXT NOT NULL UNIQUE
    );

    CREATE TABLE hechos_capacidad_renovable (
        id_pais INTEGER NOT NULL REFERENCES dim_pais(id_pais),
        anio INTEGER NOT NULL CHECK (anio BETWEEN 2000 AND 2021),
        capacidad_renovable_total_w_persona REAL,
        capacidad_renovable_bioenergia_w_persona REAL,
        capacidad_renovable_geotermica_w_persona REAL,
        capacidad_renovable_hidroelectrica_w_persona REAL,
        capacidad_renovable_marina_w_persona REAL,
        capacidad_renovable_solar_w_persona REAL,
        capacidad_renovable_eolica_w_persona REAL,
        PRIMARY KEY (id_pais, anio)
    );

    CREATE TABLE hechos_capacidad (
        id_pais INTEGER NOT NULL REFERENCES dim_pais(id_pais),
        anio INTEGER NOT NULL CHECK (anio BETWEEN 2000 AND 2021),
        capacidad_eolica REAL,
        capacidad_geotermia REAL,
        capacidad_hidro REAL,
        capacidad_nuclear REAL,
        capacidad_otros REAL,
        capacidad_solar REAL,
        capacidad_termica_no_renovable REAL,
        capacidad_termica_renovable REAL,
        PRIMARY KEY (id_pais, anio)
    );

    CREATE TABLE hechos_consumo_renovable (
        id_pais INTEGER NOT NULL REFERENCES dim_pais(id_pais),
        anio INTEGER NOT NULL CHECK (anio BETWEEN 2000 AND 2021),
        consumo_renovable_electricidad_pj REAL,
        consumo_renovable_aumento_calor_pj REAL,
        consumo_renovable_transporte_pj REAL,
        PRIMARY KEY (id_pais, anio)
    );

    CREATE TABLE hechos_consumo (
        id_pais INTEGER NOT NULL REFERENCES dim_pais(id_pais),
        anio INTEGER NOT NULL CHECK (anio BETWEEN 2000 AND 2021),
        consumo_agro_pesca_y_mineria REAL,
        consumo_comercial_servicios_publico REAL,
        consumo_construccion_y_otros REAL,
        consumo_industrial REAL,
        consumo_residencial REAL,
        consumo_transporte REAL,
        consumo_final_total_energia_pj REAL,
        PRIMARY KEY (id_pais, anio)
    );

    CREATE TABLE hechos_generacion (
        id_pais INTEGER NOT NULL REFERENCES dim_pais(id_pais),
        anio INTEGER NOT NULL CHECK (anio BETWEEN 2000 AND 2021),
        generacion_eolica REAL,
        generacion_geotermia REAL,
        generacion_hidro REAL,
        generacion_nuclear REAL,
        generacion_otros REAL,
        generacion_solar REAL,
        generacion_termica_no_renovable REAL,
        generacion_termica_renovable REAL,
        PRIMARY KEY (id_pais, anio)
    );

    CREATE TABLE hechos_financieros (
        id_pais INTEGER NOT NULL REFERENCES dim_pais(id_pais),
        anio INTEGER NOT NULL CHECK (anio BETWEEN 2000 AND 2021),
        flujos_financieros_renovables_total_usd REAL,
        flujos_financieros_bioenergia_usd REAL,
        flujos_financieros_geotermica_usd REAL,
        flujos_financieros_hidroelectrica_usd REAL,
        flujos_financieros_multiples_renovables_usd REAL,
        flujos_financieros_marina_usd REAL,
        flujos_financieros_solar_usd REAL,
        flujos_financieros_eolica_usd REAL,
        PRIMARY KEY (id_pais, anio)
    );

    CREATE TABLE hechos_otros (
      id_pais INTEGER NOT NULL REFERENCES dim_pais(id_pais),
      anio INTEGER NOT NULL CHECK (anio BETWEEN 2000 AND 2021),
      acceso_electricidad_pt REAL,
      acceso_combustibles_tecnologias_limpias_pt REAL,
      proporcion_energias_renovables_modernas_pt REAL,
      proporcion_energias_renovables_total_pt REAL,
      intensidad_energetica_mj_pib REAL,
      PRIMARY KEY (id_pais, anio)
    );

    CREATE INDEX idx_capacidad_renovable_anio ON hechos_capacidad_renovable(anio);
    CREATE INDEX idx_capacidad_anio ON hechos_capacidad(anio);
    CREATE INDEX idx_generacion_anio ON hechos_generacion(anio);
    CREATE INDEX idx_consumo_renovable_anio ON hechos_consumo_renovable(anio);
    CREATE INDEX idx_consumo_anio ON hechos_consumo(anio);
    CREATE INDEX idx_flujos_financieros_anio ON hechos_financieros(anio);
    CREATE INDEX idx_otros_anio ON hechos_otros(anio);
"""
)

print("Esquema creado")

Esquema creado


## 3. Cargar los datos

Primero los países, con id en orden alfabético. Después los hechos, cambiando el nombre del país por su id.

In [13]:
paises = sorted(df["pais"].unique())
dim = pd.DataFrame({"pais": paises})
dim.to_sql("dim_pais", con, if_exists="append", index=False)

# Leer los ID que asignó la base de datos
dim_db = pd.read_sql("SELECT id_pais, pais FROM dim_pais", con)

# Unir el id_pais al DataFrame original
df_merged = df.merge(dim_db, on="pais")

# Definir la lista de columnas para cada tabla de hechos
cols_capacidad_renovable = [
    "id_pais",
    "anio",
    "capacidad_renovable_total_w_persona",
    "capacidad_renovable_bioenergia_w_persona",
    "capacidad_renovable_geotermica_w_persona",
    "capacidad_renovable_hidroelectrica_w_persona",
    "capacidad_renovable_marina_w_persona",
    "capacidad_renovable_solar_w_persona",
    "capacidad_renovable_eolica_w_persona",
]

cols_capacidad = [
    "id_pais",
    "anio",
    "capacidad_eolica",
    "capacidad_geotermia",
    "capacidad_hidro",
    "capacidad_nuclear",
    "capacidad_otros",
    "capacidad_solar",
    "capacidad_termica_no_renovable",
    "capacidad_termica_renovable",
]

cols_consumo_renovable = [
    "id_pais",
    "anio",
    "consumo_renovable_electricidad_pj",
    "consumo_renovable_aumento_calor_pj",
    "consumo_renovable_transporte_pj",
]

cols_consumo = [
    "id_pais",
    "anio",
    "consumo_agro_pesca_y_mineria",
    "consumo_comercial_servicios_publico",
    "consumo_construccion_y_otros",
    "consumo_industrial",
    "consumo_residencial",
    "consumo_transporte",
    "consumo_final_total_energia_pj",
]

cols_generacion = [
    "id_pais",
    "anio",
    "generacion_eolica",
    "generacion_geotermia",
    "generacion_hidro",
    "generacion_nuclear",
    "generacion_otros",
    "generacion_solar",
    "generacion_termica_no_renovable",
    "generacion_termica_renovable",
]

cols_financieros = [
    "id_pais",
    "anio",
    "flujos_financieros_renovables_total_usd",
    "flujos_financieros_bioenergia_usd",
    "flujos_financieros_geotermica_usd",
    "flujos_financieros_hidroelectrica_usd",
    "flujos_financieros_multiples_renovables_usd",
    "flujos_financieros_marina_usd",
    "flujos_financieros_solar_usd",
    "flujos_financieros_eolica_usd",
]

cols_otros = [
    "id_pais",
    "anio",
    "acceso_electricidad_pt",
    "acceso_combustibles_tecnologias_limpias_pt",
    "proporcion_energias_renovables_modernas_pt",
    "proporcion_energias_renovables_total_pt",
    "intensidad_energetica_mj_pib",
]

# Insertar los datos
df_merged[cols_capacidad_renovable].to_sql("hechos_capacidad_renovable", con, if_exists="append", index=False)
df_merged[cols_capacidad].to_sql("hechos_capacidad", con, if_exists="append", index=False)
df_merged[cols_consumo_renovable].to_sql("hechos_consumo_renovable", con, if_exists="append", index=False)
df_merged[cols_consumo].to_sql("hechos_consumo", con, if_exists="append", index=False)
df_merged[cols_generacion].to_sql("hechos_generacion", con, if_exists="append", index=False)
df_merged[cols_financieros].to_sql("hechos_financieros", con, if_exists="append", index=False)
df_merged[cols_otros].to_sql("hechos_otros", con, if_exists="append", index=False)
con.commit()

# Verificar la cantidad de registros insertados
print(f"dim_pais: {len(dim_db)} filas")
print(f"hechos_capacidad_renovable: {pd.read_sql('SELECT COUNT(*) FROM hechos_capacidad_renovable', con).iloc[0,0]} filas")
print(f"hechos_capacidad: {pd.read_sql('SELECT COUNT(*) FROM hechos_capacidad', con).iloc[0,0]} filas")
print(f"hechos_consumo_renovable: {pd.read_sql('SELECT COUNT(*) FROM hechos_consumo_renovable', con).iloc[0,0]} filas")
print(f"hechos_consumo: {pd.read_sql('SELECT COUNT(*) FROM hechos_consumo', con).iloc[0,0]} filas")
print(f"hechos_generacion: {pd.read_sql('SELECT COUNT(*) FROM hechos_generacion', con).iloc[0,0]} filas")
print(f"hechos_financieros: {pd.read_sql('SELECT COUNT(*) FROM hechos_financieros', con).iloc[0,0]} filas")
print(f"hechos_otros: {pd.read_sql('SELECT COUNT(*) FROM hechos_otros', con).iloc[0,0]} filas")

dim_pais: 20 filas
hechos_capacidad_renovable: 440 filas
hechos_capacidad: 440 filas
hechos_consumo_renovable: 440 filas
hechos_consumo: 440 filas
hechos_generacion: 440 filas
hechos_financieros: 440 filas
hechos_otros: 440 filas


## 4. Validar la carga

Comparo lo que hay en la base de datos contra el CSV. Si algo no cuadra, el `assert` del final corta la ejecución.

In [14]:
def q(sql):
    return con.execute(sql).fetchone()[0]

# Lista de tablas de hechos creadas
tablas_hechos = ["hechos_capacidad_renovable", "hechos_capacidad", "hechos_consumo", "hechos_consumo_renovable",
                 "hechos_generacion", "hechos_financieros", "hechos_otros",]

# Validar filas por cada tabla
filas_por_tabla = [q(f"SELECT COUNT(*) FROM {t}") for t in tablas_hechos]

n_paises = q("SELECT COUNT(*) FROM dim_pais")

# Buscar duplicados acumulados en todas las tablas de hechos
duplicados = sum(q(f"SELECT COUNT() FROM (SELECT id_pais, anio FROM {t} GROUP BY id_pais, anio HAVING COUNT() > 1)") for t in tablas_hechos)

# Nulos en llaves en todas las tablas
nulos_llave = sum(q(f"SELECT COUNT(*) FROM {t} WHERE id_pais IS NULL OR anio IS NULL") for t in tablas_hechos)

# Registros huérfanos sin país en dim_pais
huerfanos = sum(q(f"SELECT COUNT(*) FROM {t} h LEFT JOIN dim_pais p USING (id_pais) WHERE p.id_pais IS NULL") for t in tablas_hechos)

# Rango de años
anio_min, anio_max = con.execute("SELECT MIN(anio), MAX(anio) FROM hechos_capacidad").fetchone()

resumen = pd.DataFrame(
    [
        ("Filas por tabla de hechos", len(df), f"{filas_por_tabla[0]} (en {len(tablas_hechos)} tablas)", ),
        ("Países", df["pais"].nunique(), n_paises),
        ("Duplicados (país, año)", int(df.duplicated(["pais", "anio"]).sum()), duplicados, ),
        ("Nulos en llaves", int(df[["pais", "anio"]].isna().sum().sum()), nulos_llave, ),
        ("Hechos sin país (huérfanos)", "n/a", huerfanos),
        ("Rango de años", f"{df['anio'].min()}-{df['anio'].max()}", f"{anio_min}-{anio_max}", ),
    ],
    columns=["Verificación", "CSV", "SQLite"],
)
display(resumen)

# 8. Assert de integridad
ok = (
    all(f == len(df) for f in filas_por_tabla)
    and n_paises == df["pais"].nunique()
    and duplicados == 0
    and nulos_llave == 0
    and huerfanos == 0
)
assert ok, "La validación falló, revisa la tabla de arriba"
print("Validación OK")

,Verificación,CSV,SQLite
0,Filas por tabla de hechos,440,440 (en 7 tablas)
1,Países,20,20
2,"Duplicados (país, año)",0,0
3,Nulos en llaves,0,0
4,Hechos sin país (huérfanos),n/a,0
5,Rango de años,2000-2021,2000-2021


Validación OK


## 5. Consultas de prueba

Unas consultas para ver que todo responde.

In [15]:
# Países con más generación hidroeléctrica en 2021
pd.read_sql("""
    SELECT p.pais, ROUND(hg.generacion_hidro, 2) AS generacion_hidro
    FROM hechos_generacion hg
    JOIN dim_pais p USING (id_pais)
    WHERE hg.anio = 2021
    ORDER BY hg.generacion_hidro DESC
    LIMIT 5
""", con)

,pais,generacion_hidro
0,brasil,1306.15
1,venezuela,244.24
2,colombia,217.79
3,paraguay,146.07
4,mexico,124.98


In [16]:
# Promedio 2000-2021 de la proporción de renovables en el consumo final
pd.read_sql("""
    SELECT p.pais,
           ROUND(AVG(ho.proporcion_energias_renovables_total_pt), 2) AS prom_renovables_pct
    FROM hechos_otros ho
    JOIN dim_pais p USING (id_pais)
    GROUP BY p.pais
    ORDER BY prom_renovables_pct DESC
    LIMIT 5
""", con)

,pais,prom_renovables_pct
0,haiti,78.86
1,guatemala,64.73
2,paraguay,64.27
3,nicaragua,52.76
4,honduras,50.10


In [17]:
# Capacidad de energia solar e hidro para Colombia despues del 2015
pd.read_sql("""
    SELECT pais, anio, capacidad_solar, capacidad_hidro
    FROM hechos_capacidad hc
    INNER JOIN dim_pais p USING (id_pais)
    WHERE pais like '%colombia%' AND anio >= 2015
    ORDER BY anio
""", con)

,pais,anio,capacidad_solar,capacidad_hidro
0,colombia,2015,0.000000,362.681282
1,colombia,2016,0.000000,366.019367
2,colombia,2017,0.309053,369.779405
3,colombia,2018,0.309053,373.278008
4,colombia,2019,0.566885,375.802150
5,colombia,2020,1.916736,376.690961
6,colombia,2021,4.288581,376.690961


In [18]:
con.close()
print("Base lista en:", DB)

Base lista en: /content/drive/My Drive/Colab Notebooks/Master/1.ETL/entregable3/data/processed/energia.db
